In [1]:
import pandas as pd
import numpy as np
import time
from scipy.optimize import minimize, Bounds

# Load input data
data = pd.read_excel('data/input42u.xlsx')
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

penalty_factor = 1e6

# Load power demand
power_demand_data = pd.read_excel('data/power_demand42u.xlsx')
power_demand = power_demand_data['load'].values

# Cost function with penalty
def calculate_fuel_cost_with_penalty(P, a, b, c, demand):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.divide(np.add(a, np.multiply(b, P) + np.multiply(c, squaredP)), 1000)
    power_difference = np.sum(P) - demand
    penalty_term = penalty_factor * abs(power_difference)
    total_Fuel_Cost = np.sum(Fuel_Cost) + penalty_term
    return total_Fuel_Cost

# BFGS optimization wrapper
def bfgs(func, initial_guess, bounds):
    result = minimize(
        fun=func,
        x0=initial_guess,
        bounds=bounds,
        method='BFGS'
    )
    return result.x, result.fun

# Main optimization loop
all_outputs = []
schedulling = []

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values

    initial_guess = (Minimum_Capacity + Maximum_Capacity) / 2
    bounds = list(zip(Minimum_Capacity, Maximum_Capacity))

    def objective(P):
        return calculate_fuel_cost_with_penalty(P, a, b, c, demand)

    P_opt, cost = bfgs(objective, initial_guess, bounds)

    # Enforce capacity and power balance
    for _ in range(10):
        P_opt = np.minimum(np.maximum(P_opt, Minimum_Capacity), Maximum_Capacity)
        power_difference = np.sum(P_opt) - demand
        if power_difference > 0:
            for j in range(len(P_opt)):
                if P_opt[j] > Minimum_Capacity[j]:
                    reduction = min(P_opt[j] - Minimum_Capacity[j], power_difference)
                    P_opt[j] -= reduction
                    power_difference -= reduction
        elif power_difference < 0:
            for j in range(len(P_opt)):
                if P_opt[j] < Maximum_Capacity[j]:
                    increase = min(Maximum_Capacity[j] - P_opt[j], abs(power_difference))
                    P_opt[j] += increase
                    power_difference += increase

    end_time = time.time()

    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P_opt})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P_opt):.2f} MW")
    print(f'Total Fuel Cost: Rp {cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    all_outputs.append({
        'Demand': demand,
        'Output': output,
        'Total Power Produced': np.sum(P_opt),
        'Total Fuel Cost': cost,
        'Computation Time': end_time - start_time
    })

    output_info = {'Demand': demand}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = P_opt[unit_index]
        output_info[f'Unit {unit} Cost'] = cost
    schedulling.append(output_info)

# Ramp rate and generation limit check
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):
    prev = all_outputs[i - 1]['Output']['Power Produced (MW)'].values
    curr = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = curr - prev
    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)

    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = violations[unit_index]
        gen_violation = (curr[unit_index] < Minimum_Capacity[unit_index]) or (curr[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = curr[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = gen_violation
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)

# Save to Excel
output_file_path = 'BFGS (ED).xlsx'
with pd.ExcelWriter(output_file_path) as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(all_outputs).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)

print("All results saved to 'BFGS (ED).xlsx'")

/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(


Load Counter: 1
Output for Power Demand 8761.42 MW
    Unit  Power Produced (MW)
0      1           371.500000
1      2           371.500000
2      3           261.625763
3      4           250.000000
4      5           408.000000
5      6           408.000000
6      7           408.000000
7      8           174.600000
8      9           174.600000
9     10           174.600000
10    11           174.600000
11    12            40.000000
12    13            40.000000
13    14            75.000000
14    15            75.000000
15    16           220.000000
16    17           220.000000
17    18           220.000000
18    19            10.698820
19    20            10.699416
20    21           118.500000
21    22           118.500000
22    23           118.500000
23    24           118.500000
24    25           118.500000
25    26           118.500000
26    27           118.500000
27    28           118.500000
28    29           400.000000
29    30           400.000000
30    31           

/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(


Load Counter: 6
Output for Power Demand 8462.08 MW
    Unit  Power Produced (MW)
0      1           250.000000
1      2           250.000000
2      3           250.000000
3      4           250.000000
4      5           408.000000
5      6           408.000000
6      7           408.000000
7      8           130.615428
8      9           174.600000
9     10           174.600000
10    11           174.600000
11    12            40.000000
12    13            40.000000
13    14            75.000000
14    15            75.000000
15    16           220.000000
16    17           220.000000
17    18           220.000000
18    19            10.332477
19    20            10.332094
20    21           118.500000
21    22           118.500000
22    23           118.500000
23    24           118.500000
24    25           118.500000
25    26           118.500000
26    27           118.500000
27    28           118.500000
28    29           400.000000
29    30           400.000000
30    31           

/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(


Load Counter: 10
Output for Power Demand 8759.57 MW
    Unit  Power Produced (MW)
0      1           371.500000
1      2           371.500000
2      3           259.776693
3      4           250.000000
4      5           408.000000
5      6           408.000000
6      7           408.000000
7      8           174.600000
8      9           174.600000
9     10           174.600000
10    11           174.600000
11    12            40.000000
12    13            40.000000
13    14            75.000000
14    15            75.000000
15    16           220.000000
16    17           220.000000
17    18           220.000000
18    19            10.698664
19    20            10.696644
20    21           118.500000
21    22           118.500000
22    23           118.500000
23    24           118.500000
24    25           118.500000
25    26           118.500000
26    27           118.500000
27    28           118.500000
28    29           400.000000
29    30           400.000000
30    31          

/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(


Load Counter: 18
Output for Power Demand 10023.51 MW
    Unit  Power Produced (MW)
0      1           371.500000
1      2           371.500000
2      3           371.500000
3      4           371.500000
4      5           575.000000
5      6           575.000000
6      7           575.000000
7      8           174.600000
8      9           174.600000
9     10           174.600000
10    11           174.600000
11    12            80.000000
12    13            80.000000
13    14           170.000000
14    15           170.000000
15    16           476.756248
16    17           220.000000
17    18           220.000000
18    19            11.989255
19    20            12.364498
20    21           118.500000
21    22           118.500000
22    23           118.500000
23    24           118.500000
24    25           118.500000
25    26           118.500000
26    27           118.500000
27    28           118.500000
28    29           400.000000
29    30           400.000000
30    31         

/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(


Load Counter: 22
Output for Power Demand 10411.10 MW
    Unit  Power Produced (MW)
0      1           371.500000
1      2           371.500000
2      3           371.500000
3      4           371.500000
4      5           575.000000
5      6           575.000000
6      7           575.000000
7      8           174.600000
8      9           174.600000
9     10           174.600000
10    11           174.600000
11    12            80.000000
12    13            80.000000
13    14           170.000000
14    15           170.000000
15    16           480.000000
16    17           480.000000
17    18           343.276497
18    19            12.711890
19    20            12.711613
20    21           118.500000
21    22           118.500000
22    23           118.500000
23    24           118.500000
24    25           118.500000
25    26           118.500000
26    27           118.500000
27    28           118.500000
28    29           400.000000
29    30           400.000000
30    31         

/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
/var/folders/th/s6s5rhk92_319jmpt7c9n_cw0000gn/T/ipykernel_2311/953477298.py:34: RuntimeWarning: Method BFGS cannot handle bounds.
  result = minimize(
